In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

## Multi-Representation Indexing

![Multi-Representation Indexing Image](Images/Multi-representation_Indexing.png)

In [5]:
import uuid
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers import MultiVectorRetriever

In [2]:
# Load the Raw Documents 

print("--- 1. Loading Documents ---")

loader1 = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader1.load()

loader2 = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader2.load())

print(f"Total documents loaded: {len(docs)}")
print(f"Type of documents: {type(docs[0])}\n")

--- 1. Loading Documents ---
Total documents loaded: 2
Type of documents: <class 'langchain_core.documents.base.Document'>



In [3]:
# Generate Summaries (The "Bait")

print("--- 2. Generating Summaries ---")

# We use a simple chain to ask the LLM to summarize the page content
chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document concisely:\n\n{doc}")
    | ChatOpenAI(model="gpt-4o-mini", max_retries=0)
    | StrOutputParser()
)

# .batch() processes all documents in parallel
summaries = chain.batch(docs, {"max_concurrency": 5})

print(f"Generated {len(summaries)} summaries.")
print(f"Type of summary output: {type(summaries[0])}")
print(f"Preview of first summary: {summaries[0][:150]}...\n")

--- 2. Generating Summaries ---
Generated 2 summaries.
Type of summary output: <class 'langchain_core.messages.base.TextAccessor'>
Preview of first summary: The document by Lilian Weng explores the development of LLM (Large Language Model)-powered autonomous agents, outlining their essential components: pl...



In [6]:
# Setup the Databases and Retriever 

print("--- 3. Setting up Multi-Vector Retriever ---")

# 3a. Vectorstore (To hold the embeddings of the summaries)
vectorstore = Chroma(
    collection_name="summaries",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small")
)

# 3b. Docstore (To hold the raw, full-text parent documents)
# Note: Replaced InMemoryByteStore with InMemoryStore to handle Document objects directly
store = InMemoryStore()
id_key = "doc_id"

# 3c. The Retriever that links them together
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)

--- 3. Setting up Multi-Vector Retriever ---


In [8]:
# Link Summaries to Parent Docs and Index them

print("--- 4. Linking and Indexing ---")

# Generate a unique ID for each document
doc_ids = [str(uuid.uuid4()) for _ in docs]
print(f"Generated Document IDs: {doc_ids}")

# Create Document objects for the summaries, embedding the parent ID in the metadata
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

print(f"Type of summary_docs: {type(summary_docs[0])}")
print(f"Metadata injected into summary: {summary_docs[0].metadata}")

# Add the summaries to the Vector Database
retriever.vectorstore.add_documents(summary_docs)

# Add the full original documents to the Document Store (mapped to the same IDs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

print("Successfully added summaries to Vectorstore and full docs to Docstore.\n")

--- 4. Linking and Indexing ---
Generated Document IDs: ['334998e4-88dd-422f-9f6e-ab690a2c87c7', '05d9f685-bc4d-4b53-90b4-de780535b4c1']
Type of summary_docs: <class 'langchain_core.documents.base.Document'>
Metadata injected into summary: {'doc_id': '334998e4-88dd-422f-9f6e-ab690a2c87c7'}
Successfully added summaries to Vectorstore and full docs to Docstore.



In [9]:
# Execution and Comparison 

print("--- 5. Retrieval Comparison ---")
query = "Memory in agents"

# A. Direct Vectorstore Search (This only hits the summaries)
print("\n>>> A. Direct Vectorstore Search (Hits Summary Only):")
sub_docs = vectorstore.similarity_search(query, k=1)

print(f"Type returned: {type(sub_docs[0])}")
print(f"Metadata: {sub_docs[0].metadata}")
print(f"Content length: {len(sub_docs[0].page_content)} characters")
print(f"Content Preview:\n{sub_docs[0].page_content[:300]}...\n")

# B. Multi-Vector Retriever Search (The "Switch" - Returns the Parent Doc)
# Note: Modern LangChain uses .invoke() instead of .get_relevant_documents()
print("\n>>> B. Multi-Vector Retriever Search (Returns Full Parent Document):")
retrieved_docs = retriever.invoke(query)

# We slice the list because the Retriever might return multiple docs based on its configuration, 
# but since we only have 2 docs in the DB, it will likely return the best one.
best_doc = retrieved_docs[0]

print(f"Type returned: {type(best_doc)}")
print(f"Metadata: {best_doc.metadata}") # Notice the doc_id is NOT here, this is the original metadata!
print(f"Content length: {len(best_doc.page_content)} characters")
print(f"Content Preview:\n{best_doc.page_content[:300]}...\n")

--- 5. Retrieval Comparison ---

>>> A. Direct Vectorstore Search (Hits Summary Only):
Type returned: <class 'langchain_core.documents.base.Document'>
Metadata: {'doc_id': '334998e4-88dd-422f-9f6e-ab690a2c87c7'}
Content length: 1530 characters
Content Preview:
The document by Lilian Weng explores the development of LLM (Large Language Model)-powered autonomous agents, outlining their essential components: planning, memory, and tool use. 

1. **Agent System Overview**: LLM serves as the brain for these agents, enabling them to decompose complex tasks into ...


>>> B. Multi-Vector Retriever Search (Returns Full Parent Document):
Type returned: <class 'langchain_core.documents.base.Document'>
Metadata: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer an

## RAPTOR (Recursive Abstractive Processing for Tree-Organized Retrieval)

![RAPTOR Architecture Diagram](Images/Raptor_Image.png)

Deep dive video:

https://www.youtube.com/watch?v=jbGchdTL7d0

Full code:

https://github.com/langchain-ai/langchain/blob/master/cookbook/RAPTOR.ipynb

### RAPTOR (Recursive Abstractive Processing for Tree-Organized Retrieval) is a retrieval method that organizes and summarizes documents in a hierarchical tree structure to improve search and answer generation.

## ColBERT (Contextualized Late Interaction over BERT)

### ColBERT (Contextualized Late Interaction over BERT) is a retrieval model that uses token-level embeddings and late interaction to efficiently match queries with documents for high-precision search.

In [8]:
import requests
from ragatouille import RAGPretrainedModel
import transformers
import colbert.modeling.colbert
import colbert.search.index_storage
import colbert.search.strided_tensor
import torch

In [2]:
# Fetch the Raw Document (Wikipedia)

print("--- 1. Fetching Wikipedia Document ---")

def get_wikipedia_page(title: str):
    """Retrieve the full text content of a Wikipedia page."""
    
    URL = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
    }
    
    # Custom User-Agent header to comply with Wikipedia's best practices
    headers = {"User-Agent": "RAG_Learning_Env/0.0.1"}

    response = requests.get(URL, params=params, headers=headers)
    data = response.json()
    
    page = next(iter(data["query"]["pages"].values()))
    return page.get("extract", None)

full_document = get_wikipedia_page("Quantum_computing")

print(f"Type of fetched document: {type(full_document)}")
print(f"Total length: {len(full_document)} characters")
print(f"Document Preview: {full_document[:500]}...\n")

--- 1. Fetching Wikipedia Document ---
Type of fetched document: <class 'str'>
Total length: 58644 characters
Document Preview: A quantum computer is a (real or theoretical) computer that exploits superposed and entangled states. Quantum computers can be viewed as sampling from quantum systems. These systems evolve in ways that operate on an enormous number of possibilities simultaneously, though they remain subject to strict computational constraints. By contrast, ordinary ("classical") computers operate according to deterministic rules. (A classical computer can, in principle, be replicated by a classical mechanical de...



In [3]:
# Load the ColBERT model 

print("--- 2. Loading ColBERT Model ---")

# Check if we already patched it so we don't create an infinite loop!
if not hasattr(transformers.PreTrainedModel, "_is_patched_for_colbert"):
    
    # Monkey-patch for Transformers compatibility
    original_mark_tied = transformers.PreTrainedModel.mark_tied_weights_as_initialized

    def safe_mark_tied(self, *args, **kwargs):
        
        # If the older ColBERT model forgot to declare this variable, do it for them
        if not hasattr(self, 'all_tied_weights_keys'):
            self.all_tied_weights_keys = {}
            
        return original_mark_tied(self, *args, **kwargs)

    # Overwrite the strict function with our safe, patched version
    transformers.PreTrainedModel.mark_tied_weights_as_initialized = safe_mark_tied
    
    # Mark as patched
    transformers.PreTrainedModel._is_patched_for_colbert = True

# Bypasses the missing 'cl.exe' compiler error by forcing ColBERT to use standard PyTorch
colbert.modeling.colbert.ColBERT.try_load_torch_extensions = classmethod(lambda cls, *args, **kwargs: None)

# This downloads the pre-trained ColBERT weights (can take a minute on first run)
RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

print(f"Type of RAG model: {type(RAG)}\n")

--- 2. Loading ColBERT Model ---


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

HF_ColBERT LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Type of RAG model: <class 'ragatouille.RAGPretrainedModel.RAGPretrainedModel'>



c:\Users\ashut\anaconda3\envs\RAG_env\Lib\site-packages\colbert\utils\amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
c:\Users\ashut\anaconda3\envs\RAG_env\Lib\site-packages\torch\cuda\amp\grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


In [4]:
# Indexing (Token-Level Chunking & Embedding)

print("--- 3. Indexing the Document ---")

print("ColBERT is currently breaking the document into sub-word tokens and creating a vector for EACH token...")

# RAG.index actually saves a physical index on your hard drive (usually in a .ragatouille folder)
index_path = RAG.index(
    collection=[full_document],
    index_name="Quantum-123",
    max_document_length=180, # ColBERT's version of chunking: max tokens per chunk
    split_documents=True,
)

print(f"Indexing complete! Index saved to disk at: {index_path}\n")

--- 3. Indexing the Document ---
ColBERT is currently breaking the document into sub-word tokens and creating a vector for EACH token...
---- WARNING! You are using PLAID with an experimental replacement for FAISS for greater compatibility ----
This is a behaviour change from RAGatouille 0.8.0 onwards.
This works fine for most users and smallish datasets, but can be considerably slower than FAISS and could cause worse results in some situations.
If you're confident with FAISS working on your machine, pass use_faiss=True to revert to the FAISS-using behaviour.
--------------------


[Mar 06, 14:38:24] #> Creating directory .ragatouille/colbert\indexes/Quantum-123 




Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

HF_ColBERT LOAD REPORT from: colbert-ir/colbertv2.0
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Mar 06, 14:38:28] [0] 		 #> Encoding 86 passages..


  0%|          | 0/3 [00:00<?, ?it/s]c:\Users\ashut\anaconda3\envs\RAG_env\Lib\site-packages\colbert\utils\amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
c:\Users\ashut\anaconda3\envs\RAG_env\Lib\site-packages\torch\cuda\amp\autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
100%|██████████| 3/3 [00:09<00:00,  3.03s/it]

...

[Mar 06, 14:38:38] [0] 		 #> Encoding 86 passages..


100%|██████████| 3/3 [00:08<00:00,  2.94s/it]
1it [00:08,  8.89s/it]
100%|██████████| 1/1 [00:00<00:00, 73.27it/s]

[Mar 06, 14:38:47] #> Optimizing IVF to store map from centroids to list of pids..
[Mar 06, 14:38:47] #> Building the emb2pid mapping..
[Mar 06, 14:38:47] len(emb2pid) = 10167



100%|██████████| 1024/1024 [00:00<00:00, 57341.92it/s]

[Mar 06, 14:38:47] #> Saved optimized IVF to .ragatouille/colbert\indexes/Quantum-123\ivf.pid.pt
Done indexing!
Indexing complete! Index saved to disk at: .ragatouille\colbert\indexes\Quantum-123



In [ ]:
# Native Search (Seeing the raw ColBERT math) 
# This doesn't work!!

print("--- 4. Native RAGatouille Search ---")

# Stop it from trying to compile the C++ files
colbert.search.index_storage.IndexScorer.try_load_torch_extensions = classmethod(lambda cls, *args, **kwargs: None)
colbert.search.strided_tensor.StridedTensor.try_load_torch_extensions = classmethod(lambda cls, *args, **kwargs: None)

# Provide a pure PyTorch fallback for the missing C++ lookup function
def pure_pytorch_lookup(tensor, pids, lengths, offsets):
    """Fallback for segmented_lookup_cpp"""
    # Flatten the tensor based on offsets and lengths
    output = []
    for pid, length, offset in zip(pids, lengths, offsets):
        output.append(tensor[offset:offset+length])
    return torch.cat(output) if output else torch.tensor([], dtype=tensor.dtype, device=tensor.device)

# Provide a pure Python fallback for filter_pids
def pure_pytorch_filter_pids(pids, centroid_scores, codes, doclens, offsets, idx, ndocs):
    """Fallback for filter_pids_cpp"""
    
    if len(pids) > ndocs:
        return pids[:ndocs]
    return pids

# Attach the fallback to the class so it finds it!
colbert.search.strided_tensor.StridedTensor.segmented_lookup = pure_pytorch_lookup
colbert.search.index_storage.IndexScorer.filter_pids = staticmethod(pure_pytorch_filter_pids)

query = "Who is Peter Shor?"
print(f"Query: '{query}'\n")

# This searches the index and returns the raw Python dictionaries
raw_results = RAG.search(query=query, k=2) # We'll just look at the top 2 for brevity

print(f"Type of raw_results: {type(raw_results)}")
print(f"Type of a single result: {type(raw_results[0])}")
print(f"Keys inside a result: {raw_results[0].keys()}\n")

print(">>> Top Raw Result Details:")
print(f"Rank: {raw_results[0]['rank']}")
print(f"Document ID: {raw_results[0]['document_id']}")

# This score is the sum of the Maximum Similarities (MaxSim) of all the query tokens matched against the chunk's tokens!
print(f"ColBERT MaxSim Score: {raw_results[0]['score']}") 
print(f"Content: {raw_results[0]['content']}...\n")

--- 4. Native RAGatouille Search ---
Query: 'Who is Peter Shor?'



AttributeError: type object 'IndexScorer' has no attribute 'decompress_residuals'

### Trying the normal method instead of ColBERT which doesn't work

In [2]:
import requests
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [4]:
# Fetch the Raw Document (Wikipedia)

print("--- 1. Fetching Wikipedia Document ---")

def get_wikipedia_page(title: str):
    """Retrieve the full text content of a Wikipedia page."""
    URL = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
    }
    headers = {"User-Agent": "RAG_Learning_Env/0.0.1"}

    response = requests.get(URL, params=params, headers=headers)
    data = response.json()
    
    page = next(iter(data["query"]["pages"].values()))
    return page.get("extract", None)

raw_text = get_wikipedia_page("Quantum_computing")

print(f"Type of fetched text: {type(raw_text)}")
print(f"Total length: {len(raw_text)} characters\n")
print(f"Document Preview: {raw_text[:500]}...\n")

--- 1. Fetching Wikipedia Document ---
Type of fetched text: <class 'str'>
Total length: 58644 characters

Document Preview: A quantum computer is a (real or theoretical) computer that exploits superposed and entangled states. Quantum computers can be viewed as sampling from quantum systems. These systems evolve in ways that operate on an enormous number of possibilities simultaneously, though they remain subject to strict computational constraints. By contrast, ordinary ("classical") computers operate according to deterministic rules. (A classical computer can, in principle, be replicated by a classical mechanical de...



In [5]:
# Chunking (Since we aren't using ColBERT's internal chunker)

print("--- 2. Chunking the Text ---")

# We must manually wrap our string in a LangChain Document object
doc = Document(page_content=raw_text, metadata={"source": "Wikipedia: Quantum_computing"})

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200
)
chunks = text_splitter.split_documents([doc])

print(f"Type of chunks list: {type(chunks)}")
print(f"Type of a single chunk: {type(chunks[0])}")
print(f"Total chunks created: {len(chunks)}\n")

--- 2. Chunking the Text ---
Type of chunks list: <class 'list'>
Type of a single chunk: <class 'langchain_core.documents.base.Document'>
Total chunks created: 97



In [6]:
# Embed and Index into ChromaDB

print("--- 3. Indexing into ChromaDB ---")

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# This creates a temporary Chroma database in memory
vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings_model,
    collection_name="quantum_db"
)

print(f"Type of vectorstore: {type(vectorstore)}")
print("Successfully embedded all chunks and loaded them into ChromaDB!\n")

--- 3. Indexing into ChromaDB ---
Type of vectorstore: <class 'langchain_community.vectorstores.chroma.Chroma'>
Successfully embedded all chunks and loaded them into ChromaDB!



In [7]:
# The Retriever

print("--- 4. Setting up the Retriever ---")

# Convert the vectorstore into a retriever that fetches the top 3 results
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Type of retriever: {type(retriever)}\n")

--- 4. Setting up the Retriever ---
Type of retriever: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>



In [ ]:
# Raw Search, Full RAG Chain later 

print("\n>>> A. Raw Search:")

query = "Who is Peter Shor and what did he do?"

# A. Just testing the raw retrieval
print("\n>>> A. Raw Retrieval Results:")
retrieved_docs = retriever.invoke(query)

print(f"Number of docs retrieved: {len(retrieved_docs)}")
print(f"Type of first retrieved doc: {type(retrieved_docs[0])}")
print(f"Preview of top matching chunk:\n{retrieved_docs[0].page_content[:200]}...\n")


>>> A. Raw Retrieval Results:
Number of docs retrieved: 3
Type of first retrieved doc: <class 'langchain_core.documents.base.Document'>
Preview of top matching chunk:
on the difficulty of factoring integers or the discrete logarithm problem, both of which can be solved by Shor's algorithm. In particular, the RSA, Diffie–Hellman, and elliptic curve Diffie–Hellman al...



In [9]:
# Full Rag Chain

print("\n>>> B. Full RAG Generation:")

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    """Helper to extract text from Document objects for the prompt"""
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

final_answer = rag_chain.invoke(query)

print(f"Type of final answer: {type(final_answer)}")
print(f"Final Answer:\n{final_answer}")


>>> B. Full RAG Generation:
Type of final answer: <class 'langchain_core.messages.base.TextAccessor'>
Final Answer:
Peter Shor is a researcher known for his contributions to quantum computing, particularly for developing Shor's algorithm in 1994. This algorithm is significant because it can efficiently factor integers and solve the discrete logarithm problem, which are foundational to widely used encryption protocols like RSA and Diffie–Hellman. His work drew considerable attention to the field of quantum computing due to its implications for breaking these encryption methods, thereby impacting electronic privacy and security.
